# Project Patho · Process and review a whole slide

**Workflow:** upload your original SVS/NDPI/pyramidal TIFF to your private Google Drive → run this notebook → choose the file path below → **Process slide** → open the generated review report. There is no manual result-JSON import step. Your laptop displays results; Colab performs inference.

The report includes the slide overview, heatmap, raw score and threshold class for each evaluated support, uncalibrated model certainty, spatial means, connected regions and a slide-wide **research flag**. It does **not** provide a validated whole-slide diagnosis or confidence. A maximum tile score is not a slide probability.

The original is read in batches at nominal PCam scale (0.972 µm per pixel); the model input is 96×96 RGB divided by 255, with central 32×32 supports. Source slide MPP metadata is required and must be verified. The tissue filter is heuristic: excluded supports remain unscored. Full grid coverage is checked explicitly. This version stops before starting if the grid exceeds 5 million supports. Colab can disconnect: export your report and keep the checkpoint files.

The report embeds a maximum-1400px overview, not original-resolution slide tiles. Keep the original slide and local workstation for native-resolution review. All results remain private unless you share them. No paid cloud server or public upload endpoint is created.

Use only deidentified slides you are authorized to process in Colab/Drive. The public reference sample is skin and is provided for technical checks only.


In [ ]:
# 1. Connect Drive. If mounting fails, rerun this cell and complete Google's prompt.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/Pathology_Project')
MODEL_PATH = PROJECT / 'my_cancer_model.h5'
assert MODEL_PATH.is_file(), 'Set MODEL_PATH to the existing saved model in your Drive.'
print('Saved model located. Existing model and pilot files will not be changed.')


In [ ]:
# 2. Install the slide reader. TensorFlow and SciPy normally exist in Colab.
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','openslide-python==1.4.7','openslide-bin==4.0.1.2','scipy'])
import tensorflow as tf, numpy as np
print('TensorFlow',tf.__version__, 'GPU:',tf.config.list_physical_devices('GPU'))


In [ ]:
# 3. Shared input-processing helpers
"""Bounded, inference-only slide processing and browser review export."""
import base64
import csv
import datetime
import hashlib
import io
import json
import math
import platform
import time
from pathlib import Path

import numpy as np
import openslide
from PIL import Image

def digest(path):
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for block in iter(lambda:stream.read(4*1024*1024),b''):
            h.update(block)
    return h.hexdigest()

def scale_for(slide,target_mpp,purpose):
    if purpose=='technical_smoke_test' and target_mpp is None:
        return 1.0
    if purpose not in ('research','technical_smoke_test') or not isinstance(target_mpp,(int,float)) or not math.isfinite(target_mpp) or target_mpp<=0:
        raise ValueError('Research runs require a confirmed target_mpp in microns per model pixel.')
    try:
        x=float(slide.properties['openslide.mpp-x']);y=float(slide.properties['openslide.mpp-y'])
    except (KeyError,ValueError):
        raise ValueError('Slide has no usable pixel scale; verify its scanner metadata first.')
    if not all(math.isfinite(v) and v>0 for v in (x,y)) or abs(x-y)/x>.01:
        raise ValueError('This version requires verified, approximately square source pixels.')
    scale=target_mpp/x
    if not .25<=scale<=32:
        raise ValueError('Requested scale is outside this implementation’s supported bounds.')
    return scale

def context(slide,x,y,scale):
    """x,y are central-support origin in analysis pixels; pad context white."""
    level=slide.get_best_level_for_downsample(scale)
    native=slide.level_downsamples[level]
    extent=max(1,math.ceil(96*scale/native))
    rgba=slide.read_region((round((x-32)*scale),round((y-32)*scale)),level,(extent,extent))
    background=Image.new('RGBA',rgba.size,'white')
    image=Image.alpha_composite(background,rgba).convert('RGB').resize((96,96),Image.Resampling.BILINEAR)
    return np.asarray(image,dtype=np.uint8)

def tissue_fraction(image):
    center=image[32:64,32:64].astype(np.int16)
    return float(((center.mean(axis=2)<235)&((center.max(axis=2)-center.min(axis=2))>12)).mean())

def process_slide(slide_path,predictor,output_dir,*,model_sha256,model_name,purpose='technical_smoke_test',target_mpp=None,max_candidates=2000,batch_size=16,min_tissue=.1,roi=None,source_slide_id=None):
    if not 1<=max_candidates<=100000 or not 1<=batch_size<=32 or not 0<=min_tissue<=1:
        raise ValueError('Invalid candidate, batch or tissue limit.')
    if not isinstance(model_sha256,str) or len(model_sha256)!=64:
        raise ValueError('Record the actual saved model SHA-256.')
    source=Path(slide_path)
    slide=openslide.OpenSlide(str(source))
    try:
        slide.set_cache(openslide.OpenSlideCache(64*1024*1024))
        scale=scale_for(slide,target_mpp,purpose)
        width,height=slide.dimensions
        aw,ah=math.ceil(width/scale),math.ceil(height/scale)
        cols,rows=math.ceil(aw/32),math.ceil(ah/32)
        if rows*cols>10**10:
            raise ValueError('Slide grid exceeds the viewer contract.')
        if roi is None: roi=[0,0,aw,ah]
        if len(roi)!=4 or not all(isinstance(v,int) for v in roi):
            raise ValueError('ROI is [x,y,width,height] in analysis pixels, using integers.')
        rx,ry,rw,rh=roi
        if rx<0 or ry<0 or rw<=0 or rh<=0 or rx+rw>aw or ry+rh>ah or rx%32 or ry%32:
            raise ValueError('ROI must start on the 32 px grid and stay within the analysis image.')
        source_hash=digest(source)
        out=Path(output_dir);out.mkdir(parents=True,exist_ok=False)
        started=time.perf_counter();patches=[];pending=[];images=[];examined=0;skipped=0
        csv_path=out/'predictions.csv'
        fields=['id','row','col','x','y','width','height','p','tissue_fraction']
        config={'purpose':purpose,'target_mpp':target_mpp,'coordinate_scale':scale,'roi_analysis_pixels':roi,'max_candidates':max_candidates,'batch_size':batch_size,'min_tissue':min_tissue,'source_sha256':source_hash,'model_sha256':model_sha256,'preprocessing':'RGB uint8 to float32 / 255.0','context_size':96,'support_size':32,'stride':32,'sampling_order':'row-major within ROI; candidate cap includes skipped background'}
        (out/'run_config.json').write_text(json.dumps(config,indent=2))
        with csv_path.open('x',newline='') as stream:
            writer=csv.DictWriter(stream,fieldnames=fields);writer.writeheader()
            def flush():
                if not images:return
                batch=np.stack(images).astype(np.float32)/255.0
                predictions=np.asarray(predictor(batch))
                if predictions.shape not in ((len(images),),(len(images),1)):
                    raise ValueError('Expected one sigmoid probability per patch.')
                values=predictions.reshape(-1)
                if not np.all(np.isfinite(values)) or np.any(values<0) or np.any(values>1):
                    raise ValueError('Model output is not finite probability data.')
                for patch,value in zip(pending,values):
                    patch['p']=float(value);patches.append(patch);writer.writerow(patch)
                stream.flush();images.clear();pending.clear()
            try:
                for row in range(ry//32,math.ceil((ry+rh)/32)):
                    for col in range(rx//32,math.ceil((rx+rw)/32)):
                        if examined>=max_candidates:break
                        examined+=1;x,y=col*32,row*32
                        image=context(slide,x,y,scale);fraction=tissue_fraction(image)
                        if fraction<min_tissue:skipped+=1;continue
                        pending.append({'id':f'r{row}c{col}','row':row,'col':col,'x':x,'y':y,'width':min(32,aw-x),'height':min(32,ah-y),'tissue_fraction':round(fraction,4)})
                        images.append(image)
                        if len(images)>=batch_size:flush()
                    if examined>=max_candidates:break
                flush()
            except Exception:
                (out/'FAILED.txt').write_text('Run incomplete. predictions.csv contains only completed batches; do not treat it as a complete slide evaluation.')
                raise
        thumbnail=slide.get_thumbnail((1400,1400)).convert('RGB');buffer=io.BytesIO();thumbnail.save(buffer,'JPEG',quality=85)
        thumbnail_uri='data:image/jpeg;base64,'+base64.b64encode(buffer.getvalue()).decode()
        def mpp(key):
            try:
                value=float(slide.properties.get(key,''))
                return value if value>0 and math.isfinite(value) else None
            except ValueError:return None
        mx,my=mpp('openslide.mpp-x'),mpp('openslide.mpp-y')
        roi_cells=(math.ceil((rx+rw)/32)-rx//32)*(math.ceil((ry+rh)/32)-ry//32)
        coverage={'examined_candidates':examined,'scored_supports':len(patches),'background_skipped':skipped,'roi_candidates':roi_cells,'roi_scan_complete':examined==roi_cells,'whole_slide_scan_complete':roi==[0,0,aw,ah] and examined==roi_cells,'tissue_rule':'heuristic central-support brightness/chroma filter; not validated tissue segmentation'}
        sid=source_slide_id or 'sha256:'+source_hash
        result={'schema_version':1,'mode':'colab','purpose':purpose,'source_slide_id':sid,'source_sha256':source_hash,'coordinate_scale':scale,'slide':{'id':sid,'name':source.name,'width':aw,'height':ah,'mpp_x':mx*scale if mx else None,'mpp_y':my*scale if my else None},'viewer_slide':{'id':sid,'name':source.name,'width':width,'height':height,'levels':slide.level_count,'vendor':slide.properties.get('openslide.vendor','Unknown'),'mpp_x':mx,'mpp_y':my,'objective':None,'bytes':source.stat().st_size,'overview':thumbnail_uri},'grid':{'rows':rows,'cols':cols,'stride':32},'model':{'name':model_name,'sha256':model_sha256,'input_size':96,'support_size':32,'preprocessing':'RGB / 255.0'},'coverage':coverage,'patches':patches,'diagnosis_generated':False}
        payload=json.dumps(result,separators=(',',':'),allow_nan=False)
        if len(payload.encode())>16_000_000:
            raise ValueError('Result exceeds 16 MB browser limit. Use a smaller ROI or candidate cap; CSV remains saved.')
        (out/'dashboard-result.json').write_text(payload)
        manifest={**config,**coverage,'elapsed_seconds':round(time.perf_counter()-started,2),'utc_completed':datetime.datetime.now(datetime.timezone.utc).isoformat(),'python':platform.python_version(),'numpy':np.__version__,'openslide':openslide.__library_version__,'predictions_sha256':digest(csv_path),'overview_only_in_dashboard':True,'diagnosis_generated':False}
        (out/'manifest.json').write_text(json.dumps(manifest,indent=2))
        return out/'dashboard-result.json',manifest
    finally:
        slide.close()


In [ ]:
# 4. Full-grid processing and spatial summaries
"""Full-grid streamed inference, explicit coverage and exploratory slide summary."""
import base64
import csv
import datetime
import io
import json
import math
import time
from pathlib import Path
import numpy as np
import openslide
from scipy import ndimage

# In a standalone notebook the shared helpers are defined in an earlier cell.
try:
    from pipeline import digest,scale_for,context,tissue_fraction
except ImportError:
    pass

def summarize_grid(raw,*,threshold=.5,min_cells=4,complete=False):
    if not 0<=threshold<=1 or not isinstance(min_cells,int) or min_cells<1:
        raise ValueError('Invalid fixed research-summary rules.')
    valid=np.isfinite(raw)
    sums=np.zeros(raw.shape,dtype=np.float32);counts=np.zeros(raw.shape,dtype=np.uint8)
    for dy in (-1,0,1):
        for dx in (-1,0,1):
            sy=slice(max(0,-dy),min(raw.shape[0],raw.shape[0]-dy));sx=slice(max(0,-dx),min(raw.shape[1],raw.shape[1]-dx))
            ty=slice(max(0,dy),min(raw.shape[0],raw.shape[0]+dy));tx=slice(max(0,dx),min(raw.shape[1],raw.shape[1]+dx))
            values=np.asarray(raw[sy,sx]);mask=valid[sy,sx]
            sums[ty,tx]+=np.where(mask,values,0);counts[ty,tx]+=mask
    spatial=np.full(raw.shape,np.nan,dtype=np.float32)
    np.divide(sums,counts,out=spatial,where=valid&(counts>0))
    active=valid&(spatial>=threshold)
    labels,n=ndimage.label(active,structure=np.ones((3,3),dtype=np.uint8))
    sizes=np.bincount(labels.ravel());ids=np.flatnonzero(sizes>=min_cells);ids=ids[ids!=0]
    ids=ids[np.argsort(sizes[ids])[::-1]]
    boxes=ndimage.find_objects(labels)
    regions=[]
    for ident in ids[:100]:
        bounds=boxes[int(ident)-1]
        regions.append({'id':len(regions)+1,'supports':int(sizes[ident]),'row_start':bounds[0].start,'row_stop':bounds[0].stop,'col_start':bounds[1].start,'col_stop':bounds[1].stop})
    finite=np.asarray(raw[valid])
    summary={'scored_supports':int(valid.sum()),'raw_above_threshold':int((valid&(raw>=threshold)).sum()),'raw_max':float(finite.max()) if finite.size else None,'raw_mean':float(finite.mean()) if finite.size else None,'threshold':threshold,'minimum_connected_supports':min_cells,'candidate_regions':len(ids),'largest_region_supports':int(sizes[ids[0]]) if len(ids) else 0,'summary_complete':bool(complete),'slide_probability':None,'diagnosis':None,'confidence_calibrated':False,'rule':'3x3 mean of existing scored neighbors, then 8-connected components; not trained slide classification'}
    summary['research_flag']='INCOMPLETE — no slide-wide interpretation' if not complete else 'NO SCORED SUPPORTS — indeterminate' if not finite.size else 'Candidate evidence for review' if len(ids) else 'No candidate region at these rules — not a negative diagnosis'
    return spatial,summary,regions

def process_full_slide(slide_path,predictor,output_dir,*,model_sha256,model_name,target_mpp=.972,purpose='research',batch_size=16,min_tissue=.1,threshold=.5,min_cells=4,max_candidates=None,progress=None):
    if not 1<=batch_size<=32 or not 0<=min_tissue<=1:
        raise ValueError('Invalid batch or tissue settings.')
    if max_candidates is not None and (not isinstance(max_candidates,int) or max_candidates<1):
        raise ValueError('Candidate cap must be a positive integer or None for full scan.')
    source=Path(slide_path);slide=openslide.OpenSlide(str(source))
    out=Path(output_dir)
    try:
        scale=scale_for(slide,target_mpp,purpose);slide.set_cache(openslide.OpenSlideCache(64*1024*1024))
        w,h=slide.dimensions;aw,ah=math.ceil(w/scale),math.ceil(h/scale);cols,rows=math.ceil(aw/32),math.ceil(ah/32)
        total=cols*rows
        if total>5_000_000 or max(cols,rows)>16384:
            raise ValueError('Grid exceeds this report version’s 5-million-support/16384-axis resource limit. No partial slide summary will be substituted.')
        source_hash=digest(source);out.mkdir(parents=True,exist_ok=False)
        raw=np.lib.format.open_memmap(out/'raw_scores.npy',mode='w+',dtype=np.float32,shape=(rows,cols));raw[:]=np.nan;raw.flush()
        cfg={'source_sha256':source_hash,'model_sha256':model_sha256,'model_name':model_name,'purpose':purpose,'target_mpp':target_mpp,'coordinate_scale':scale,'analysis_width':aw,'analysis_height':ah,'cols':cols,'rows':rows,'total_supports':total,'batch_size':batch_size,'min_tissue':min_tissue,'threshold':threshold,'min_cells':min_cells,'max_candidates':max_candidates,'preprocessing':'RGB float32 /255.0; 96x96 context; central 32x32 support; inference training=False'}
        (out/'run_config.json').write_text(json.dumps(cfg,indent=2))
        start=time.perf_counter();examined=0;skipped=0;scored=0;images=[];pending=[]
        with (out/'predictions.csv').open('x',newline='') as stream:
            writer=csv.writer(stream);writer.writerow(['row','col','analysis_x','analysis_y','support_width','support_height','tumor_score','threshold_class','uncalibrated_model_certainty'])
            def checkpoint():
                raw.flush();stream.flush()
                (out/'checkpoint.json').write_text(json.dumps({'complete':False,'examined':examined,'scored':scored,'background_skipped':skipped,'total':total,'elapsed_seconds':round(time.perf_counter()-start,2)}))
                if progress:progress(examined,total,scored)
            def flush():
                nonlocal scored
                if not images:return
                values=np.asarray(predictor(np.stack(images).astype(np.float32)/255.0))
                if values.shape not in ((len(images),),(len(images),1)) or not np.all(np.isfinite(values)) or np.any(values<0) or np.any(values>1):
                    raise ValueError('Expected one finite sigmoid probability per patch.')
                for (row,col),value in zip(pending,values.reshape(-1)):
                    p=float(value);raw[row,col]=p;writer.writerow([row,col,col*32,row*32,min(32,aw-col*32),min(32,ah-row*32),p,'above_threshold' if p>=threshold else 'below_threshold',max(p,1-p)]);scored+=1
                images.clear();pending.clear();checkpoint()
            try:
                limit=total if max_candidates is None else min(total,max_candidates)
                for index in range(limit):
                    row,col=divmod(index,cols);image=context(slide,col*32,row*32,scale);examined+=1
                    if tissue_fraction(image)<min_tissue:skipped+=1
                    else:images.append(image);pending.append((row,col))
                    if len(images)>=batch_size:flush()
                    if examined%1000==0:checkpoint()
                flush();checkpoint()
            except Exception:
                checkpoint();(out/'FAILED.txt').write_text('Incomplete processing. CSV contains completed inference batches only. No finished slide summary was generated.');raise
        complete=examined==total
        spatial,summary,regions=summarize_grid(raw,threshold=threshold,min_cells=min_cells,complete=complete)
        np.save(out/'spatial_scores.npy',spatial)
        summary.update({'purpose':purpose,'grid_scan_complete':complete,'total_supports':total,'examined_supports':examined,'background_excluded':skipped,'tissue_filter_validated':False,'tile_probability_calibrated':False})
        if purpose=='technical_smoke_test':summary['research_flag']='TECHNICAL TEST — not model evaluation evidence'
        (out/'slide_summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False))
        (out/'regions.json').write_text(json.dumps(regions,indent=2))
        thumb=slide.get_thumbnail((1400,1400)).convert('RGB');buffer=io.BytesIO();thumb.save(buffer,'JPEG',quality=85)
        viewer={'source_sha256':source_hash,'model_sha256':model_sha256,'name':source.name,'width':w,'height':h,'analysis_width':aw,'analysis_height':ah,'cols':cols,'rows':rows,'coordinate_scale':scale,'thumbnail':'data:image/jpeg;base64,'+base64.b64encode(buffer.getvalue()).decode(),'summary':summary,'regions':regions}
        manifest={**cfg,**summary,'utc_completed':datetime.datetime.now(datetime.timezone.utc).isoformat(),'elapsed_seconds':round(time.perf_counter()-start,2),'predictions_sha256':digest(out/'predictions.csv'),'diagnosis_generated':False,'report_uses_overview_only':True}
        (out/'manifest.json').write_text(json.dumps(manifest,indent=2,allow_nan=False));(out/'checkpoint.json').write_text(json.dumps({'complete':complete,'examined':examined,'total':total,'scored':scored}))
        return viewer,raw,spatial,manifest
    finally:slide.close()

def write_review_report(path,viewer,raw,spatial,template):
    data=dict(viewer)
    data['raw']=base64.b64encode(np.asarray(raw,dtype='<f4').tobytes()).decode()
    data['spatial']=base64.b64encode(np.asarray(spatial,dtype='<f4').tobytes()).decode()
    # Escape '<' so filenames or metadata cannot terminate the JSON script block.
    payload=json.dumps(data,separators=(',',':'),allow_nan=False).replace('<','\\u003c')
    Path(path).write_text(template.replace('__REPORT_JSON__',payload),encoding='utf-8')
    return Path(path)


In [ ]:
# 5. Load your saved model for inference only.
MODEL_HASH = digest(MODEL_PATH)
model = tf.keras.models.load_model(MODEL_PATH,compile=False)
assert tuple(model.input_shape[1:])==(96,96,3), f'Unexpected input shape: {model.input_shape}'
assert tuple(model.output_shape[1:])==(1,), f'Unexpected output shape: {model.output_shape}'
def predictor(batch): return model(batch,training=False).numpy()
assert np.isfinite(predictor(np.zeros((1,96,96,3),np.float32))).all()
print('Model ready; original file hash:',MODEL_HASH)


In [ ]:
# 6. Choose a source and process. For a real slide, upload it to Google Drive first.
import ipywidgets as widgets
from IPython.display import display,HTML,clear_output
from google.colab import files
import urllib.request,uuid
REPORT_TEMPLATE = '<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Patho · Slide review report</title><style>\n*{box-sizing:border-box}body{margin:0;background:#111720;color:#d0dbe8;font:13px \'Segoe UI\',Arial,sans-serif}header{padding:17px 24px;border-bottom:1px solid #384456;display:flex;align-items:center;gap:20px;background:#1d2633}header b{letter-spacing:2px}header span{color:#9cafc5;font-size:12px}button,select,input,textarea{font:inherit;color:inherit}button,select{background:#26384d;border:1px solid #4c617c;padding:8px 11px;border-radius:4px;cursor:pointer}button:hover{background:#355271}.layout{display:grid;grid-template-columns:minmax(0,1fr) 310px;min-height:calc(100vh - 58px)}main{padding:20px;min-width:0}aside{padding:22px;background:#1b2533;border-left:1px solid #384456;overflow-wrap:anywhere}h1{font-size:18px;margin:0 0 8px}h2{font-size:13px;margin:22px 0 10px}.muted{color:#96a8bf;font-size:11px;line-height:1.7}.notice{padding:13px;background:#382e21;color:#e4cfa7;border:1px solid #6e5736;border-radius:5px;font-size:12px;line-height:1.6;margin:15px 0}.metrics{display:grid;grid-template-columns:repeat(3,1fr);gap:10px;margin:16px 0}.metric{padding:14px;background:#202d3e;border:1px solid #37495f;border-radius:4px}.metric b{display:block;font-size:22px;margin-top:6px}.metric small{color:#a2b1c5}.toolbar{display:flex;align-items:center;gap:8px;flex-wrap:wrap;padding:10px;background:#243145}.toolbar label{font-size:11px;color:#b9c8dc}input[type=range]{width:85px;accent-color:#6bb5d9}#stage{height:57vh;min-height:340px;background:#090e15;position:relative;overflow:hidden}canvas{width:100%;height:100%;display:block;touch-action:none;cursor:grab}#coords{padding:9px 12px;background:#202b3b;font-size:11px;color:#acbfd8}.row{font-size:12px;border-bottom:1px solid #344256;padding:9px 0;display:flex;justify-content:space-between;gap:12px}.row span{color:#98abc3}.row b{text-align:right;font-weight:550}#flag{line-height:1.6;font-size:14px}#regions{max-height:180px;overflow:auto}.region{display:block;width:100%;text-align:left;font-size:11px;margin:7px 0}.region small{display:block;margin-top:5px;color:#adc0d5}textarea{width:100%;min-height:90px;resize:vertical;background:#101c2b;border:1px solid #455b75;border-radius:4px;padding:10px;margin:8px 0}footer{font-size:11px;color:#91a4bf;line-height:1.7;padding:15px 0}#loading{padding:40px;color:#cad9eb}button:focus-visible,select:focus-visible,input:focus-visible{outline:2px solid #9bd9f6;outline-offset:2px}@media(max-width:850px){.layout{grid-template-columns:1fr}aside{border:0}.metrics{grid-template-columns:repeat(3,1fr)}header{gap:10px;padding:14px}#stage{height:60vh}main{padding:12px}}\n</style></head><body><header><b>PATHO</b><span>Whole-slide research review</span><button id="export" style="margin-left:auto">Export review notes</button></header><div id="loading">Loading score grids…</div><div class="layout" id="app" hidden><main><h1 id="name"></h1><div class="muted" id="metadata"></div><div class="notice" id="notice"></div><div class="metrics"><div class="metric"><small>Scored supports</small><b id="scored"></b></div><div class="metric"><small>Above raw threshold</small><b id="positive"></b></div><div class="metric"><small>Candidate regions</small><b id="nregions"></b></div></div><div class="toolbar"><select id="layer" aria-label="Display layer"><option value="raw">Raw tumor scores</option><option value="spatial">Spatial means</option><option value="none">Slide overview</option></select><label>Opacity <input id="opacity" type="range" min="0" max="100" value="55"></label><button id="minus" aria-label="Zoom out">−</button><button id="plus" aria-label="Zoom in">+</button><button id="fit">Fit</button><button id="csv">Download all tile scores</button></div><div id="stage"><canvas id="view" aria-label="Slide overview and score map. Drag to pan, scroll to zoom, click a support to inspect."></canvas></div><div id="coords">Click a support to inspect its prediction.</div><footer>This report displays an overview image, not original-resolution slide tiles. Raw model outputs are not calibrated confidence. Spatial means and connected regions are exploratory. Background-excluded supports remain unscored. Keep the original slide for pathology review.</footer></main><aside><h2 style="margin-top:0">Slide-level research summary</h2><p id="flag"></p><div class="row"><span>Slide diagnosis</span><b>Not validated / not generated</b></div><div class="row"><span>Slide confidence</span><b>Not estimated</b></div><div class="row"><span>Grid scan coverage</span><b id="coverage"></b></div><div class="row"><span>Background excluded</span><b id="excluded"></b></div><div class="row"><span>Highest raw tile score</span><b id="max"></b></div><div class="row"><span>Fixed review threshold</span><b id="threshold"></b></div><h2>Candidate regions</h2><div id="regions"></div><p class="muted">3×3 means, 8-neighbor connectivity, minimum <span id="mincells"></span> supports. Regions do not establish lesions or disease stage.</p><h2>Selected support</h2><div id="patch" class="muted">Select any location. Unscored background is distinguished from a below-threshold prediction.</div><h2>Review notes</h2><textarea id="notes" placeholder="Observations for mentor review…"></textarea><p class="muted">Export notes before closing. Nothing in this report is uploaded.</p></aside></div>\n<script type="application/json" id="report-data">__REPORT_JSON__</script><script>\n\'use strict\';const d=JSON.parse(document.getElementById(\'report-data\').textContent);document.getElementById(\'report-data\').remove();const $=s=>document.querySelector(s),fmt=n=>n.toLocaleString();\nfunction decode(text){const bin=atob(text),bytes=new Uint8Array(bin.length);for(let i=0;i<bin.length;i++)bytes[i]=bin.charCodeAt(i);return new DataView(bytes.buffer)}\nconst raw=decode(d.raw),spatial=decode(d.spatial);delete d.raw;delete d.spatial;\nconst score=(arr,i)=>arr.getFloat32(i*4,true),summary=d.summary,canvas=$(\'#view\'),ctx=canvas.getContext(\'2d\'),heat=document.createElement(\'canvas\');heat.width=d.cols;heat.height=d.rows;\nlet zoom=1,panX=0,panY=0,selected=null,drag=null,layer=\'raw\',opacity=.55,scheduled=false;const image=new Image();\n$(\'#name\').textContent=d.name;$(\'#metadata\').textContent=`${fmt(d.width)} × ${fmt(d.height)} original pixels · ${fmt(d.cols)} × ${fmt(d.rows)} support grid · 96 px input / central 32 px support`;\n$(\'#notice\').textContent=summary.purpose===\'technical_smoke_test\'?\'TECHNICAL TEST ONLY — this run verifies processing and display. It is not evidence of diagnostic performance.\':summary.grid_scan_complete?\'The full support grid was examined. Only tissue-filtered supports were scored. This is an unvalidated research review, not a diagnosis.\':\'INCOMPLETE COVERAGE — do not interpret these findings as a whole-slide assessment.\';\n$(\'#scored\').textContent=fmt(summary.scored_supports);$(\'#positive\').textContent=fmt(summary.raw_above_threshold);$(\'#nregions\').textContent=fmt(summary.candidate_regions);$(\'#flag\').textContent=summary.research_flag;\n$(\'#coverage\').textContent=`${fmt(summary.examined_supports)} / ${fmt(summary.total_supports)}`;$(\'#excluded\').textContent=fmt(summary.background_excluded);$(\'#max\').textContent=summary.raw_max==null?\'Unavailable\':summary.raw_max.toFixed(4);$(\'#threshold\').textContent=summary.threshold.toFixed(2);$(\'#mincells\').textContent=summary.minimum_connected_supports;\nfunction transform(){const box=canvas.getBoundingClientRect(),fit=Math.min((box.width-30)/d.width,(box.height-30)/d.height);return {w:box.width,h:box.height,scale:fit*zoom,x:(box.width-d.width*fit*zoom)/2+panX,y:(box.height-d.height*fit*zoom)/2+panY}}\nfunction makeHeat(){const ac=heat.getContext(\'2d\'),pixels=ac.createImageData(d.cols,d.rows),array=layer===\'spatial\'?spatial:raw;for(let i=0;i<d.rows*d.cols;i++){const p=score(array,i);if(!Number.isFinite(p))continue;pixels.data.set([Math.round(p*255),Math.round((1-p)*110),Math.round((1-p)*255),255],i*4)}ac.putImageData(pixels,0,0)}\nfunction draw(){const t=transform(),ratio=Math.min(devicePixelRatio||1,2);if(canvas.width!==Math.round(t.w*ratio)||canvas.height!==Math.round(t.h*ratio)){canvas.width=Math.round(t.w*ratio);canvas.height=Math.round(t.h*ratio)}ctx.setTransform(ratio,0,0,ratio,0,0);ctx.clearRect(0,0,t.w,t.h);ctx.save();ctx.translate(t.x,t.y);ctx.scale(t.scale,t.scale);ctx.drawImage(image,0,0,d.width,d.height);if(layer!==\'none\'){ctx.globalAlpha=opacity;ctx.imageSmoothingEnabled=false;ctx.drawImage(heat,0,0,d.cols*32*d.coordinate_scale,d.rows*32*d.coordinate_scale);ctx.globalAlpha=1}if(selected!=null){const row=Math.floor(selected/d.cols),col=selected%d.cols;ctx.strokeStyle=\'white\';ctx.lineWidth=2/t.scale;ctx.strokeRect(col*32*d.coordinate_scale,row*32*d.coordinate_scale,Math.min(32,d.analysis_width-col*32)*d.coordinate_scale,Math.min(32,d.analysis_height-row*32)*d.coordinate_scale)}ctx.restore()}\nfunction schedule(){if(!scheduled){scheduled=true;requestAnimationFrame(()=>{scheduled=false;draw()})}}\nfunction zoomBy(mult,x=null,y=null){const t=transform(),next=Math.max(.5,Math.min(20,zoom*mult));if(x!=null){const ratio=next/zoom;panX=x-t.w/2-(x-t.w/2-panX)*ratio;panY=y-t.h/2-(y-t.h/2-panY)*ratio}zoom=next;schedule()}\n$(\'#plus\').onclick=()=>zoomBy(1.5);$(\'#minus\').onclick=()=>zoomBy(1/1.5);$(\'#fit\').onclick=()=>{zoom=1;panX=0;panY=0;schedule()};$(\'#layer\').onchange=e=>{layer=e.target.value;makeHeat();schedule()};$(\'#opacity\').oninput=e=>{opacity=+e.target.value/100;schedule()};\nfunction inspect(x,y){if(x<0||y<0||x>=d.width||y>=d.height)return;const col=Math.floor(x/d.coordinate_scale/32),row=Math.floor(y/d.coordinate_scale/32),i=row*d.cols+col,p=score(raw,i),mean=score(spatial,i);selected=i;$(\'#coords\').textContent=`Grid ${col}, ${row} · original x ${Math.round(x)}, y ${Math.round(y)}`;const patch=$(\'#patch\');patch.replaceChildren();for(const [label,value] of [[\'Raw tumor score\',Number.isFinite(p)?p.toFixed(4):\'Unscored\'],[\'Threshold class\',Number.isFinite(p)?p>=summary.threshold?\'Above threshold\':\'Below threshold\':\'Not evaluated\'],[\'Model certainty\',Number.isFinite(p)?`${(Math.max(p,1-p)*100).toFixed(1)}% — uncalibrated`:\'Unavailable\'],[\'Spatial mean\',Number.isFinite(mean)?mean.toFixed(4):\'Unavailable\']]){const el=document.createElement(\'div\');el.className=\'row\';const a=document.createElement(\'span\'),b=document.createElement(\'b\');a.textContent=label;b.textContent=value;el.append(a,b);patch.append(el)}schedule()}\ncanvas.onpointerdown=e=>{canvas.setPointerCapture(e.pointerId);drag={x:e.clientX,y:e.clientY,px:panX,py:panY,moved:false}};canvas.onpointermove=e=>{if(!drag)return;const dx=e.clientX-drag.x,dy=e.clientY-drag.y;drag.moved=drag.moved||Math.hypot(dx,dy)>3;panX=drag.px+dx;panY=drag.py+dy;schedule()};canvas.onpointerup=e=>{if(drag&&!drag.moved){const b=canvas.getBoundingClientRect(),t=transform();inspect((e.clientX-b.left-t.x)/t.scale,(e.clientY-b.top-t.y)/t.scale)}drag=null};canvas.onpointercancel=()=>drag=null;canvas.addEventListener(\'wheel\',e=>{e.preventDefault();const b=canvas.getBoundingClientRect();zoomBy(Math.exp(-e.deltaY*.0015),e.clientX-b.left,e.clientY-b.top)},{passive:false});\nfor(const r of d.regions){const button=document.createElement(\'button\');button.className=\'region\';button.textContent=`Region ${r.id} · ${fmt(r.supports)} supports`;button.onclick=()=>{const box=canvas.getBoundingClientRect(),rw=(r.col_stop-r.col_start)*32*d.coordinate_scale,rh=(r.row_stop-r.row_start)*32*d.coordinate_scale,fit=Math.min((box.width-30)/d.width,(box.height-30)/d.height);zoom=Math.max(1,Math.min(10,Math.min(box.width/(rw+200),box.height/(rh+200))/fit));panX=(d.width/2-(r.col_start+r.col_stop)*16*d.coordinate_scale)*fit*zoom;panY=(d.height/2-(r.row_start+r.row_stop)*16*d.coordinate_scale)*fit*zoom;schedule()};$(\'#regions\').append(button)}if(!d.regions.length)$(\'#regions\').textContent=\'No candidate region at the recorded rules.\';\nfunction download(name,data,type){const url=URL.createObjectURL(new Blob([data],{type})),a=document.createElement(\'a\');a.href=url;a.download=name;a.click();setTimeout(()=>URL.revokeObjectURL(url),5000)}\n$(\'#export\').onclick=()=>download(\'patho-review-notes.json\',JSON.stringify({slide_name:d.name,source_sha256:d.source_sha256,model_sha256:d.model_sha256,summary,notes:$(\'#notes\').value,diagnosis_generated:false,exported_at:new Date().toISOString()},null,2),\'application/json\');\n$(\'#csv\').onclick=()=>{const lines=[\'row,col,analysis_x,analysis_y,tumor_score,threshold_class,uncalibrated_model_certainty\'];for(let i=0;i<d.rows*d.cols;i++){const p=score(raw,i);if(!Number.isFinite(p))continue;const row=Math.floor(i/d.cols),col=i%d.cols;lines.push([row,col,col*32,row*32,p,p>=summary.threshold?\'above_threshold\':\'below_threshold\',Math.max(p,1-p)].join(\',\'))}download(\'tile-scores.csv\',lines.join(\'\\r\\n\'),\'text/csv\')};\nimage.onload=()=>{$(\'#loading\').hidden=true;$(\'#app\').hidden=false;makeHeat();new ResizeObserver(schedule).observe($(\'#stage\'));schedule()};image.onerror=()=>$(\'#loading\').textContent=\'Overview image could not be decoded.\';image.src=d.thumbnail;\n</script></body></html>\n'

source_choice=widgets.Dropdown(options=[('Public sample — technical check','sample'),('My Drive slide — research review','drive')],description='Source:')
slide_path=widgets.Text(value=str(PROJECT/'Paper_2/Slides/your_slide.svs'),description='Drive path:',layout=widgets.Layout(width='95%'))
full_scan=widgets.Checkbox(value=True,description='Examine the entire slide grid (otherwise cap at 2,000 candidates)')
run_button=widgets.Button(description='Process slide',button_style='primary',icon='play')
download_button=widgets.Button(description='Download review report',disabled=True,icon='download')
progress_bar=widgets.IntProgress(value=0,min=0,max=100,description='Progress:')
status_label=widgets.HTML(value='Upload the slide into Drive, paste its /content/drive/MyDrive/... path, then process.')
log=widgets.Output()
last_report=None
def download_report(_):
    if last_report: files.download(str(last_report))
download_button.on_click(download_report)
def run(_):
    global last_report
    run_button.disabled=True;download_button.disabled=True;last_report=None
    with log:
        clear_output(wait=True)
        try:
            if source_choice.value=='sample':
                path=Path('/content/CMU-1-Small-Region.svs')
                if not path.exists():urllib.request.urlretrieve('https://openslide.cs.cmu.edu/download/openslide-testdata/Aperio/CMU-1-Small-Region.svs',path)
                purpose='technical_smoke_test'
            else:
                path=Path(slide_path.value.strip());purpose='research'
            assert path.is_file(), f'Slide not found: {path}'
            stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
            out=PROJECT/'Paper_2/Full_Slide_Runs'/(stamp+'_'+uuid.uuid4().hex[:8])
            status_label.value='Reading patches and running the saved model in Colab…'
            def progress(done,total,scored):
                progress_bar.value=round(100*done/total)
                status_label.value=f'Examined {done:,} / {total:,} supports; scored {scored:,}. This is Colab memory, not laptop RAM.'
            viewer,raw,spatial,manifest=process_full_slide(path,predictor,out,model_sha256=MODEL_HASH,model_name=MODEL_PATH.name,target_mpp=.972,purpose=purpose,max_candidates=None if full_scan.value else 2000,progress=progress)
            assert digest(MODEL_PATH)==MODEL_HASH,'Saved model file changed unexpectedly.'
            manifest.update({'tensorflow':tf.__version__,'model_file_unchanged':True})
            (out/'manifest.json').write_text(json.dumps(manifest,indent=2))
            last_report=write_review_report(out/'review-report.html',viewer,raw,spatial,REPORT_TEMPLATE)
            raw._mmap.close();del raw,spatial
            status_label.value='Report ready. Download and open it in your browser. ' + ('Full grid examined; background exclusions remain unscored.' if manifest['grid_scan_complete'] else 'PARTIAL COVERAGE — no whole-slide interpretation.')
            download_button.disabled=False
            print('Output folder:',out)
            print('Research flag:',manifest['research_flag'])
            print('Whole-slide diagnosis: not validated; slide probability: not estimated.')
            print('Report:',last_report)
            print('CSV, raw/spatial score grids, checkpoint, region list and manifest are also saved in that folder.')
        except Exception as error:
            status_label.value='Processing did not finish. No finished slide diagnosis is available.'
            print(type(error).__name__+': '+str(error))
        finally:run_button.disabled=False
run_button.on_click(run)
display(HTML('<p><a href="https://drive.google.com/drive/folders/1scj_6QNyyHdDv90mJBM4K7p9BMYd6ScY" target="_blank">Open your Pathology_Project Drive folder to upload a slide</a></p>'))
display(source_choice,slide_path,full_scan,widgets.HBox([run_button,download_button]),progress_bar,status_label,log)
